# COGS515 Lab: Minimax and Constraint Satisfaction

**Week 5 · about 25 minutes**

> Check Python Bootcamp first if you are not proficient with Python.

**How this lab works**
- ✏️ means answer on the worksheet (pen and paper). 💻 means run a cell.
- **Predict first, then run.** Write your prediction on the worksheet *before* the cell is run.
- **No laptop?** No problem. The instructor runs this notebook on the projector, and every task can be answered on the worksheet.
- **With a laptop:** open this notebook in Google Colab (*File → Upload notebook*) or Jupyter. Nothing to install. Run the cells in order, but wait for the class at each ✏️ task.

| Part | Minutes |
|---|---|
| 0. Setup | 1 |
| 1. Minimax and alpha-beta on board Q | 11 |
| 2. Forward checking | 8 |
| 3. Wrap-up | 3 |

## 0. Setup 💻 (1 min)

Run the next cell. A **state** is a tuple of 9 squares, each `'X'`, `'O'` or `'.'` (empty), numbered like the 8-puzzle positions:

```
0 1 2
3 4 5
6 7 8
```

`TicTacToe` has the methods from last week's *Formalizing the Game Setup* slide: `player`, `actions`, `result`, `terminal` and `utility`. X moves first and is **MAX**; O is **MIN**. The utility is +1 if X has won, −1 if O has won and 0 for a draw.

In [ ]:
import math


class TicTacToe:
    LINES = ((0, 1, 2), (3, 4, 5), (6, 7, 8),     # rows
             (0, 3, 6), (1, 4, 7), (2, 5, 8),     # columns
             (0, 4, 8), (2, 4, 6))                # diagonals
    initial = tuple("." * 9)

    def player(self, state):
        """X moves first, so it is X's turn when both have played equally often."""
        return "X" if state.count("X") == state.count("O") else "O"

    def actions(self, state):
        return [square for square in range(9) if state[square] == "."]

    def result(self, state, action):
        s = list(state)
        s[action] = self.player(state)
        return tuple(s)

    def winner(self, state):
        for a, b, c in self.LINES:
            if state[a] != "." and state[a] == state[b] == state[c]:
                return state[a]
        return None

    def terminal(self, state):
        return self.winner(state) is not None or "." not in state

    def utility(self, state):
        """+1 if X has won, -1 if O has won, 0 otherwise."""
        winner = self.winner(state)
        if winner == "X":
            return 1
        if winner == "O":
            return -1
        return 0


def board(rows):
    """board("X.O/O../.X.") -> ('X', '.', 'O', 'O', '.', '.', '.', 'X', '.')"""
    return tuple(rows.replace("/", ""))


def show(state):
    """Print a state as a 3 x 3 board."""
    for row in range(3):
        print(" ".join(state[3 * row: 3 * row + 3]))      # the 3 squares of this row


game = TicTacToe()
Q = board(".OX/OX./OX.")
show(Q)

## 1. Minimax and alpha-beta on board Q ✏️ (11 min)

```
  Board Q, X to move
       . O X
       O X .
       O X .
```

Board Q has only three empty squares, so its whole game tree fits on the worksheet: 14 nodes. Squares are numbered as in Part 0; an edge labelled `X→5` means X plays square 5.

- ✏️ **Task 1a.** Write the utility (+1, 0 or −1) in the box of every **terminal** board: someone has three in a row, or the board is full.
- ✏️ **Task 1b.** Back the values up the tree. A **MAX** node (X to move) takes the *largest* value of its children; a **MIN** node (O to move) the *smallest*. What is the value of Q, and which move should X play?

Then run the three cells below to check. The first is last week's MAX-VALUE and MIN-VALUE, which you have just traced by hand. `minimax_values` prints the minimax value of each of X's moves: **+1** means X can force a win, **0** a draw with best play on both sides, **−1** that O can force a win.

In [ ]:
visited = 0    # nodes visited by the last search (global: every call below adds 1)


def max_value(game, state):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = -math.inf
    for a in game.actions(state):
        v = max(v, min_value(game, game.result(state, a)))
    return v


def min_value(game, state):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = math.inf
    for a in game.actions(state):
        v = min(v, max_value(game, game.result(state, a)))
    return v


def minimax(game, state):
    """Minimax value of a state with X (MAX) to move; also counts the nodes visited."""
    global visited
    visited = 0
    v = max_value(game, state)
    print(f"value: {v} | nodes visited: {visited:,}")
    return v


def minimax_values(game, state):
    """The minimax value of each of X's moves (X to move)."""
    for a in game.actions(state):
        print(f"X plays {a}: value {min_value(game, game.result(state, a))}")

In [ ]:
minimax_values(game, Q);

In [ ]:
minimax(game, Q);

Alpha-beta search returns the same value as minimax but skips branches that cannot change the decision. $\alpha$ is the best value MAX can already guarantee on the path to the current node, $\beta$ the best value MIN can already guarantee.

- ✏️ **Task 1c.** On the worksheet tree, cross out the nodes alpha-beta never visits. It looks at X's moves in the order 0, 5, 8 and at O's moves from left to right. *Hint:* after exploring X→0, X knows it can get at least 0, so $\alpha = 0$.
- ✏️ **Task 1d.** The next cell is last week's alpha-beta. `ab_min_value` stops looking at O's remaining moves as soon as `v <= alpha`, because MAX already has a better option elsewhere. `ab_max_value` stops looking at X's remaining moves as soon as `v >= beta`. Why can it skip them? Circle one on the worksheet:
  - (a) X has found a winning move, so nothing else can be better.
  - (b) MAX already has a better option elsewhere.
  - (c) MIN already has a better option elsewhere, so it will never let the game reach this node.
  - (d) Those moves lead to positions that were already visited.

In [ ]:
def ab_max_value(game, state, alpha, beta):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = -math.inf
    for a in game.actions(state):
        v = max(v, ab_min_value(game, game.result(state, a), alpha, beta))
        if v >= beta:                    # the cut-off on MAX's side (Task 1d)
            return v
        alpha = max(alpha, v)
    return v


def ab_min_value(game, state, alpha, beta):
    global visited
    visited += 1
    if game.terminal(state):
        return game.utility(state)
    v = math.inf
    for a in game.actions(state):
        v = min(v, ab_max_value(game, game.result(state, a), alpha, beta))
        if v <= alpha:                   # MAX already has a better option elsewhere
            return v
        beta = min(beta, v)
    return v


def alphabeta(game, state):
    """Alpha-beta value of a state with X (MAX) to move; also counts the nodes visited."""
    global visited
    visited = 0
    v = ab_max_value(game, state, -math.inf, math.inf)
    print(f"value: {v} | nodes visited: {visited:,}")
    return v

Now run alpha-beta on Q and compare the number of nodes it visits with your crossed-out tree.

In [ ]:
alphabeta(game, Q);

## 2. Forward checking (8 min)

Run the next cell.

In this lab every constraint says *two variables must take different values*, as in map colouring. A CSP is then two dictionaries:

- `domains`: each variable → the list of values it can still take (the lecture's $D$)
- `neighbors`: each variable → the variables it must differ from (the constraint graph)

The cell defines the Australia map and the lecture's forward checking. `show` prints what is left in each domain.

In [ ]:
australia = {                                  # region -> the regions it must differ from (its neighbours)
    "WA": ["NT", "SA"],
    "NT": ["WA", "SA", "Q"],
    "SA": ["WA", "NT", "Q", "NSW", "V"],
    "Q": ["NT", "SA", "NSW"],
    "NSW": ["Q", "SA", "V"],
    "V": ["SA", "NSW"],
    "T": [],
}


def forward_check(domains, neighbors, var, value):
    """Domains after var = value: value is removed from the domain of every neighbour of var."""
    new = dict(domains)                        # a copy: backtracking may need the old domains again
    new[var] = [value]
    for other in neighbors[var]:
        new[other] = [v for v in domains[other] if v != value]
    return new


def show(domains, neighbors, assigned):
    """Print each variable's remaining values and how many of its neighbours are unassigned."""
    for var in domains:
        if var in assigned:
            print(f"{var:>4}: {domains[var]}  (assigned)")
        else:
            free = 0                           # neighbours without a value yet
            for other in neighbors[var]:
                if other not in assigned:
                    free += 1
            print(f"{var:>4}: {domains[var]}  unassigned neighbours: {free}")

The worksheet shows the Australia map as a constraint graph: an edge joins two regions that must get different colours. Every region starts with the domain {red, green, blue}.

- ✏️ **Task 2a.** Assign **NSW = red**, then **NT = green**. After each assignment, apply forward checking: in the worksheet table, cross out the colours it removes.
- ✏️ **Task 2b.** Which region(s) would MRV (minimum remaining values) choose next? If more than one, which would the degree heuristic choose?

Then run the next cell to check your table. It also prints how many unassigned neighbours each region has.

In [ ]:
domains = {}                                   # every region starts with all three colours
for region in australia:
    domains[region] = ["red", "green", "blue"]
assigned = []                                  # the regions coloured so far

for region, colour in [("NSW", "red"), ("NT", "green")]:
    domains = forward_check(domains, australia, region, colour)
    assigned.append(region)
    print(f"After {region} = {colour}:")
    show(domains, australia, assigned)
    print()

- ✏️ **Task 2c.** Give the region you chose in 2b its remaining colour, and apply forward checking again. Write the new domains in the last column of the table **before** running the next cell. To run it, type your choice between the quotes.

In [ ]:
region = ""                                    # ✏️ type your choice from Task 2b between the quotes

if region in domains:
    colour = domains[region][0]                # its first remaining colour
    domains = forward_check(domains, australia, region, colour)
    assigned.append(region)
    print(f"After {region} = {colour}:")
    show(domains, australia, assigned)
else:
    print("Type a region from the table between the quotes, then run the cell again.")

## 3. Wrap-up ✏️ (3 min)

✏️ **Discussion** (in your group, then with the class)
1. Alpha-beta and forward checking both cut work. What does each one remove, and why does that never change the answer?
2. Minimax looks at every position to the end of the game before it moves. How many positions do *you* think you look at before a move?

**Next:** back to the slides: min-conflicts in practice, and how people solve these problems.